In [1]:
try:
  !pip install langchain langchain_community pypdf faiss-cpu -q
  !pip install --upgrade transformers -q
except:
  print("Langchain already installed")
import langchain
import langchain_community
import pypdf

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM , pipeline

In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter,NLTKTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline

In [4]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
def clean_text(text):
  text = text.replace("\n"," ")
  text = text.replace("  "," ")
  return text.strip()

In [7]:
def build_vectorstore(file):
    try:
      file_path = file.name if hasattr(file,"name") else file
      print("File path:",file_path)
      loader = PyPDFLoader(file.name)
      documents = loader.load()
      if not documents:
        raise ValueError("No text found in PDF")
      for doc in documents:
          doc.page_content = clean_text(doc.page_content)

      splitter = NLTKTextSplitter(
          chunk_size=500,        # measured in characters by default
          chunk_overlap=50
      )
      chunks = splitter.split_documents(documents)

      embeddings = HuggingFaceEmbeddings(
          model_name='sentence-transformers/all-MiniLM-L6-v2',
          encode_kwargs={'normalize_embeddings': True}
      )

      vectorstore = FAISS.from_documents(chunks, embeddings)

      return vectorstore
    except Exception as e:
      print("Error",e)
      return str(e)

In [8]:
vectorstore = None
retriever = None

In [9]:
def process_pdf(file):
    global vectorstore, retriever
    if file is None:
      return "Please upload a PDF"
    vectorstore = build_vectorstore(file)

    retriever = vectorstore.as_retriever(search_kwargs={'k':8})

    return "✅ PDF processed successfully!"

In [10]:
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [11]:
def solve(query,history):
    global retriever

    if retriever is None:
        return "⚠️ Please upload a PDF first."

    docs = retriever.invoke(query)

    if not docs:
        return "No documents found"

    context = ""
    sources = set()

    for doc in docs:
        context += doc.page_content + "\n"
        if "page" in doc.metadata:
            sources.add(f"Page {doc.metadata['page'] + 1}")

    context = context[:1500]

    chat_history = ""
    for q,a in history:
        chat_history += f"Q: {q}\nA: {a}\n"

    prompt = f"""
      You are a helpful AI assistant.

      Use the provided context as the primary source.
      If the context is incomplete or not helpful, use your general knowledge to answer.

      Keep the answer clear and concise (2-3 lines).

      Conversation:
      {chat_history}

      Context:
      {context}

      Question:
      {query}

      Answer:
      """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = model.generate(**inputs, max_new_tokens=150)

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if sources:
      return f"{answer}\n\nSources: {', '.join(sources)}"
    else:
      return f"{answer}\n\nSources: Model Knowledge"

In [12]:
def reset():
  global retriever,vectorstore
  retriever = None
  vectorstore = None
  return "✅ Conversation reset!"

In [13]:
import gradio as gr
with gr.Blocks(title="AI PDF Chatbot") as demo:
  gr.Markdown("Welcome to AI PDF Chatbot")
  gr.Markdown("Upload a PDF and start a conversation!")
  with gr.Row():
    file_upload = gr.File(label="Upload PDF", file_types=[".pdf"])
    process_btn = gr.Button("Process PDF")
  status = gr.Textbox()
  process_btn.click(fn=process_pdf,inputs=file_upload,outputs=status)

  chatbot = gr.ChatInterface(fn=solve)

  gr.Button("End Conversation").click(reset,outputs=status)
demo.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


KeyboardInterrupt: 